In [ ]:
%load_ext autoreload
%autoreload 3 -p

In [ ]:
import colorcet as cc
import holoviews as hv
import numpy as np
import pandas as pd
from holoviews import opts
from joblib import delayed
from omegaconf import OmegaConf
from tqdm_joblib import ParallelPbar

import gnss_tid.image
import gnss_tid.pointdata
import gnss_tid.spectral

hv.extension("bokeh")
opts.defaults(
    opts.Image(
        data_aspect=1,
        frame_height=400,
        frame_width=600,
        cmap=cc.cm.coolwarm,
        colorbar=True,
        show_grid=True,
        tools=["hover"],
    ),
    hv.opts.Scatter(
        data_aspect=1,
        frame_height=400,
        frame_width=600,
        cmap=cc.cm.coolwarm,
        colorbar=True,
        show_grid=True,
        tools=["hover"],
    ),
)

In [ ]:
# cfg = OmegaConf.load("apps/conf/event/20170820.yaml")
# cfg = OmegaConf.load("apps/conf/event/20160508.yaml")
cfg = OmegaConf.load("apps/conf/event/event_20210101.yaml")
# cfg = OmegaConf.load("apps/conf/event/20150325_full.yaml")
cfg

In [ ]:
lat_limits = cfg.latitude_limits
lon_limits = cfg.longitude_limits
time_limits = cfg.time_limits

obs, rx = gnss_tid.pointdata.load_observations(
    files=cfg.files,
    lat_limits=lat_limits,
    lon_limits=lon_limits,
    time_limits=time_limits,
    el_min=30,
)

In [ ]:
# med_stec = obs["stec"].abs().groupby(obs["rx"]).median()
# good = med_stec <= 30
# obs = obs[obs["rx"].map(good)]

In [ ]:
proj = gnss_tid.pointdata.Local2D.from_geodetic(
    np.mean(lat_limits), np.mean(lon_limits), 300.0
)
windows = gnss_tid.pointdata.make_time_windows(obs, 10, 30)
maker = gnss_tid.image.ScipyRbfImageMaker(
    hres=20,  # grid resolution (km / pixel)
    hp_freq=0.03,  # Butterworth high-pass cutoff (cycles / pixel)
    neighbor_radius=100.0,  # density-mask radius (km)
    neighbors=50,  # RBF neighborhood size
    kernel="multiquadric",
    smoothing=0.2,
)
maker.initialize_from_bounds(proj, lat_limits, lon_limits)

In [ ]:
PLOT_TYPE = "time"
N_JOBS = 8
HEIGHT = 350.0
heights = np.arange(150, 600, 10)
twin = windows[16]


def ft(t):
    return gnss_tid.pointdata.get_data(obs, rx, t, HEIGHT, lat_limits, lon_limits, proj)


def fh(h):
    return gnss_tid.pointdata.get_data(obs, rx, twin, h, lat_limits, lon_limits, proj)


if PLOT_TYPE == "time":
    dfs = ParallelPbar("making plots")(n_jobs=N_JOBS)(delayed(ft)(t) for t in windows)
    slider = [t.start_time for t in windows]
    title = f"height = {HEIGHT}km"
else:
    dfs = ParallelPbar("making plots")(n_jobs=N_JOBS)(delayed(fh)(h) for h in heights)
    slider = heights
    title = np.datetime_as_string(twin.start_time, unit="m")

In [ ]:
CLIM = 0.5

plots = {
    x: hv.Scatter(df[df["dtec3"].notna()], kdims="x", vdims=["y", "dtec3"]).opts(
        size=3.5, color="dtec3", clim=(-CLIM, CLIM), title=f"{title} | {x!s}"
    )
    for x, df in zip(slider, dfs, strict=True)
}
hv.output(widget_location="bottom")
hv.HoloMap(plots, kdims=[PLOT_TYPE])